In [1]:
import os
import re
import time
import spacy
import regex
import pickle
import string
import sqlite3
import inflect
import typing
import ahocorasick
import numpy as np
import pandas as pd
from tqdm import tqdm
from os import listdir
from itertools import islice, chain
from pandas import DataFrame
from taxonerd import TaxoNERD
from functools import lru_cache
from rich.console import Console
from os.path import isfile, join
from spacy.util import filter_spans
from IPython.display import clear_output
from spacy.tokens import Token, Doc, Span
from typing import Any, List, Dict, Tuple, Set, Optional, Union, Literal, Callable, TypedDict, Sequence

In [2]:
inflector = inflect.engine()

In [3]:
@lru_cache(maxsize=128)
def pluralize(s: Any) -> str:
    try:
        return inflector.plural(s) or s
    except Exception as e:
        return s

In [4]:
@lru_cache(maxsize=128)
def singularize(s: Any) -> str:
    try:
        return inflector.singular_noun(s) or s
    except Exception as e:
        return s

In [5]:
@lru_cache(maxsize=128)
def get_inflections(word: str) -> Set[str]:
    if not isinstance(word, str):
        return set()
    
    inflections = set()
    inflections.add(word)

    p = pluralize(word)
    if p:
        inflections.add(p)

    s = singularize(word)
    if s:
        inflections.add(s)

    return inflections

In [6]:
binomial = r'((^\p{Lu}(\.|\p{Ll}+))\s(sp\.|spp\.|(\p{Ll}(\.|\p{Ll}+))))'
trinomial = r'((^\p{Lu}(\.|\p{Ll}+))\s(\p{Ll}(\.|\p{Ll}+)\s)(sp\.|spp\.|(\p{Ll}(\.|\p{Ll}+))))'
sp = 'sp.'
spp = 'spp.'

In [7]:
def is_taxonomic_name(s: str) -> bool:
    return bool(regex.match(r'^(\p{Lu}(\.|\p{Ll}+))\s(\p{Ll}(\.|\p{Ll}+)\s)?\p{Ll}(\.|\p{Ll}+)$', s))

In [8]:
def is_taxonomic_notation(s: str) -> str:
    match = regex.match(rf'{binomial}\s((\p{{Lu}}|\(|et al|sensu|nov|comb|stat|syn|s\.s\.|s\.l\.).*)$', s)
    if match:
        return match.group(1)
    match = regex.match(rf'{trinomial}\s((\p{{Lu}}|\(|et al|sensu|nov|comb|stat|syn|s\.s\.|s\.l\.).*)$', s)
    if match:
        return match.group(1)
    return ''

In [9]:
def is_taxonomic_name_species_unknown(s: str) -> bool:
    return bool(regex.match(r'^(\p{Lu}(\.|\p{Ll}+))\s(\p{Ll}(\.|\p{Ll}+)\s)?(sp\.?|spp\.?)$', s))

In [10]:
def is_taxa(s: str) -> bool:
    return bool(regex.match(r'^\p{Lu}\p{Ll}+$', s))

In [74]:
@lru_cache(maxsize=128)
def abbreviate(s: str) -> List[str] | None:
    n = s if is_taxonomic_name(s) else is_taxonomic_notation(s)
    if not n:
        return None
    
    parts = n.split()
    abbrevs = []

    if len(parts) >= 2:
        abbrevs.append(f'{parts[0][0].upper()}. {parts[1].lower()}')
    if len(parts) == 3:
        abbrevs.append(f'{parts[0][0].upper()}. {parts[1][0].lower()}. {parts[2].lower()}')
    
    return abbrevs

In [12]:
conn = sqlite3.connect('./db/COL.db')

In [13]:
def load_data(path: str, load: Callable[[], Any], *, force: bool = False) -> Any:
    data = None
    if not force and os.path.exists(path):
        with open(path, "rb") as file:
            data = pickle.load(file)
    else:
        data = load()
        with open(path, "wb") as file:
            pickle.dump(data, file, pickle.HIGHEST_PROTOCOL)
    return data

In [14]:
def load_scientific_groups():
    df = pd.read_sql(f"""
        SELECT *
        FROM NameUsage
    """, conn)

    data = []
    cols = [
        ["kingdom", "col:kingdom"],
        ["phylum", "col:phylum"],
        ["subphylum", "col:subphylum"],
        ["class", "col:class"],
        ["subclass", "col:subclass"],
        ["order", "col:order"],
        ["suborder", "col:suborder"],
        ["superfamily", "col:superfamily"],
        ["family", "col:family"],
        ["subfamily", "col:subfamily"],
        ["genus", "col:genus"],
        ["subgenus", "col:subgenus"],
        ["genericName", "col:genericName"],
        ["specificEpithet", "col:specificEpithet"],
        ["intraspecificEpithet", "col:infraspecificEpithet"],
        ["species", "col:species"],
        ["scientificName", "col:scientificName"]
    ]
    
    for col in cols:
        values = set()
        regex = r'^([A-Za-z]+)\s\(([A-Za-z]+)\)$'
        
        for value in df[col[1]].tolist():
            if not value:
                continue

            if col[0] != "subgenus":
                values.add(value.lower())
                continue

            # The sub-genus is placed inside '(...)',
            # so we need to capture it.
            match = re.match(regex, value.lower())
            values.add(value.lower() if not match else match.group(2))
        
        data.append({
            'label': col[0],
            'column': col[1],
            'values': values
        })

    del df
    return data

scientific_groups = load_data("./db/ScientificGroups.pickle", load_scientific_groups)
print(f'# of Groups: {len(scientific_groups)}')

# of Groups: 17


In [199]:
def load_vernacular_names():
    df = pd.read_sql(f"""
        SELECT "col:name" 
        FROM VernacularName 
        WHERE "col:language" = 'eng'
    """, conn)
    
    names = set().union(*[get_inflections(name.lower()) for name in df["col:name"].tolist() if name])
    names = set(list(chain.from_iterable(names)))
    names = names | {
        # Mammals
        "mammal", "mammals",
        "rodent", "rodents",
        "primate", "primates",
        "marsupial", "marsupials",
        "bat", "bats",
        "whale", "whales",
        "dolphin", "dolphins",
        "porpoise", "porpoises",
        "seal", "seals",
        "otter", "otters",
        "bear", "bears",
        "cat", "cats",
        "dog", "dogs",
        "wolf", "wolves",
        "fox", "foxes",
        "deer", "deers",
        "antelope", "antelopes",
        "monkey", "monkeys",
        "ape", "apes",
        "shrew", "shrews",
        "mole", "moles",
        "hedgehog", "hedgehogs",
        "rabbit", "rabbits",
        "hare", "hares",
        "squirrel", "squirrels",
        "rat", "rats",
        "mouse", "mice",
        "vole", "voles",
        "beaver", "beavers",
        "pig", "pigs",
        "horse", "horses",
        "cow", "cows",
        "sheep", "sheeps",
        "goat", "goats",
        # Birds
        "bird", "birds",
        "seabird", "seabirds",
        "shorebird", "shorebirds",
        "songbird", "songbirds",
        "raptor", "raptors",
        "waterfowl",
        "wader", "waders",
        "hawk", "hawks",
        "eagle", "eagles",
        "owl", "owls",
        "parrot", "parrots",
        "pigeon", "pigeons",
        "dove", "doves",
        "duck", "ducks",
        "goose", "geese",
        "swan", "swans",
        "heron", "herons",
        "gull", "gulls",
        "tern", "terns",
        "kingfisher", "kingfishers",
        "woodpecker", "woodpeckers",
        "finch", "finches",
        "sparrow", "sparrows",
        "warbler", "warblers",
        "thrush", "thrushes",
        "robin", "robins",
        "starling", "starlings",
        "swift", "swifts",
        "swallow", "swallows",
        "martin", "martins",
        "crane", "cranes",
        "stork", "storks",
        "flamingo", "flamingos",
        "penguin", "penguins",
        "albatross", "albatrosses",
        "petrel", "petrels",
        "cormorant", "cormorants",
        "pelican", "pelicans",
        "sunbird", "sunbirds",
        "oystercatcher", "oystercatchers",
        # Reptiles
        "reptile", "reptiles",
        "snake", "snakes",
        "lizard", "lizards",
        "gecko", "geckos",
        "skink", "skinks",
        "turtle", "turtles",
        "tortoise", "tortoises",
        "crocodile", "crocodiles",
        "alligator", "alligators",
        "chameleon", "chameleons",
        # Amphibians
        "amphibian", "amphibians",
        "frog", "frogs",
        "toad", "toads",
        "salamander", "salamanders",
        "newt", "newts",
        "caecilian", "caecilians",
        # Fish
        "fish", "fishes",
        "shark", "sharks",
        "ray", "rays",
        "eel", "eels",
        "salmon", "salmons",
        "trout", "trouts",
        "cod", "cods",
        "tuna", "tunas",
        "herring", "herrings",
        "anchovy", "anchovies",
        "carp", "carps",
        "catfish", "catfishes",
        "perch", "perches",
        "bass", "basses",
        "flatfish", "flatfishes",
        "pufferfish", "pufferfishes",
        "goby", "gobies",
        # Insects
        "insect", "insects",
        "bug", "bugs",
        "beetle", "beetles",
        "fly", "flies",
        "moth", "moths",
        "butterfly", "butterflies",
        "bee", "bees",
        "wasp", "wasps",
        "ant", "ants",
        "termite", "termites",
        "dragonfly", "dragonflies",
        "damselfly", "damselflies",
        "grasshopper", "grasshoppers",
        "cricket", "crickets",
        "locust", "locusts",
        "louse", "lice",
        "flea", "fleas",
        "tick", "ticks",
        "mite", "mites",
        "aphid", "aphids",
        "cicada", "cicadas",
        "mantis", "mantises",
        "cockroach", "cockroaches",
        "earwig", "earwigs",
        "mayfly", "mayflies",
        "stonefly", "stoneflies",
        "lacewing", "lacewings",
        "weevil", "weevils",
        "moth", "moths",
        # Other Invertebrates
        "invertebrate", "invertebrates",
        "macroinvertebrate", "macroinvertebrates", 
        "microinvertebrate", "microinvertebrates", 
        "spider", "spiders",
        "scorpion", "scorpions",
        "crab", "crabs",
        "lobster", "lobsters",
        "shrimp", "shrimps",
        "prawn", "prawns",
        "barnacle", "barnacles",
        "woodlouse", "woodlice",
        "millipede", "millipedes",
        "centipede", "centipedes",
        "worm", "worms",
        "earthworm", "earthworms",
        "leech", "leeches",
        "snail", "snails",
        "slug", "slugs",
        "clam", "clams",
        "mussel", "mussels",
        "oyster", "oysters",
        "scallop", "scallops",
        "squid", "squids",
        "octopus", "octopuses",
        "cuttlefish", "cuttlefishes",
        "nautilus", "nautiluses",
        "jellyfish", "jellyfishes",
        "coral", "corals",
        "anemone", "anemones",
        "sponge", "sponges",
        "starfish", "starfishes",
        "urchin", "urchins",
        "sealion", "sealions",
        "sea lion", "sea lions",
        # Plants
        "plant", "plants",
        "tree", "trees",
        "shrub", "shrubs",
        "herb", "herbs",
        "grass", "grasses",
        "flower", "flowers",
        "weed", "weeds",
        "vine", "vines",
        "fern", "ferns",
        "moss", "mosses",
        "liverwort", "liverworts",
        "hornwort", "hornworts",
        "conifer", "conifers",
        "palm", "palms",
        "succulent", "succulents",
        "cactus", "cacti", "cactuses",
        "orchid", "orchids",
        "sedge", "sedges",
        "rush", "rushes",
        "reed", "reeds",
        "lichen", "lichens",
        "seagrass", "seagrasses",
        "seaweed", "seaweeds",
        # Fungi
        "fungus", "fungi",
        "mushroom", "mushrooms",
        "mould", "moulds",
        "mold", "molds",
        "yeast", "yeasts",
        # Microorganisms
        "bacterium", "bacteria",
        "microbe", "microbes",
        "microorganism", "microorganisms",
        "protist", "protists",
        "alga", "algae",
        "diatom", "diatoms",
        "amoeba", "amoebas", "amoebae",
        "virus", "viruses",
        "parasite", "parasites",
    }

    del df
    return names

vernacular_names = load_data("./db/VernacularNames.pickle", load_vernacular_names)
print(f'# of Names: {len(vernacular_names)}')

# of Names: 369295


In [16]:
def load_mapped_names():
    df = pd.read_sql(f'''
        SELECT *
        FROM MappedName
    ''', conn)

    m = {}
    
    for i, row in df.iterrows():
        names = [name for name in [row.ScientificName, row.VernacularName, row.Genus] if name]

        for i, name in enumerate(names):
            forms = [name.lower()]
            
            # Add Abbreviated Scientific Name
            if i == 0 and (abbrevs := abbreviate(name)):
                forms.append(abbrevs[-1].lower())

            for form in forms:
                if form not in m:
                    m[form] = set()
                
                for col in [col for col in row if col]:
                    m[form].add(col.lower())
    
    del df
    return m

mapped_names = load_data("./db/MappedNames.pickle", load_mapped_names)
print(f'# of Names: {len(mapped_names)}')

# of Names: 462190


In [216]:
role_names = {
    'carnivores',
    'carnivore',
    'predators',
    'predator',
    'species',
    'prey',
    'organism',
    'organisms',
    'competitor',
    'competitors',
    'grazer',
    'grazers',
    'producer',
    'producers',
    'consumer',
    'consumers'
}

In [217]:
def load_all_names():
    all_names = ahocorasick.Automaton()
    
    for group in scientific_groups:
        for value in group['values']:
            all_names.add_word(value, {
                'key': value,
                'label': 's',
            })
    
    for name in role_names:
        all_names.add_word(name, {
            'key': name,
            'label': 'r'
        })
    
    for name in vernacular_names:
        all_names.add_word(name, {
            'key': name,
            'label': 'v'
        })
    
    all_names.make_automaton()
    return all_names

all_names = load_data("./db/AllNames.pickle", load_all_names, force=True)
print(f'# of Names: {len(all_names)}')

# of Names: 8827243


In [ ]:
@lru_cache(maxsize=128)
def name_is_scientific(name: str) -> str:
    name = name.lower()
    for group in scientific_groups:
        if name in group['values']:
            return group['label']
    return ''

In [20]:
def name_is_role(name: str) -> bool:
    return name.lower() in role_names

In [21]:
def name_is_vernacular(name: str) -> bool:
    return name.lower() in vernacular_names

In [22]:
def name_substitutions(name: str) -> list[str]:
    return mapped_names.get(name.lower(), [])

In [ ]:
def name_is_taxa(name: str) -> bool:
    if not is_taxa(name):
        return False
    data = all_names.get(name.lower(), None)
    if not data:
        return False
    return data['label'] == 's'

In [119]:
def name_is_taxonomic(name: str) -> bool:
    if not is_taxonomic_name(name):
        return False
    data = all_names.get(name.lower(), None)
    return bool(data)

In [ ]:
def name_is_taxonomic_notation(name: str) -> str:
    taxonomic_name = is_taxonomic_notation(name)
    if not taxonomic_name:
        return ''
    data = all_names.get(taxonomic_name.lower(), None)
    if not data:
        return ''
    return taxonomic_name

In [ ]:
bacteria = {
    'escherichia coli', 
    'salmonella enterica', 
    'staphylococcus aureus', 
    'staphylococcus aureus', 
    'mycobacterium tuberculosis', 
    'clostridium botulinum', 
    'helicobacter pylori', 
    'vibrio cholerae', 
    'yersinia pestis'
}

kingdoms = {
    'chromista', 
    'helvetiavirae', 
    'bacillati', 
    'bamfordvirae', 
    'nanobdellati', 
    'sangervirae', 
    'fusobacteriati', 
    'promethearchaeati', 
    'animalia', 
    'zilligvirae', 
    'plantae', 
    'protozoa', 
    'heunggongvirae', 
    'pararnavirae', 
    'orthornavirae', 
    'shotokuvirae', 
    'trapavirae', 
    'thermotogati', 
    'fungi', 
    'thermoproteati', 
    'loebvirae', 
    'pseudomonadati', 
    'methanobacteriati', 
    'abadenavirae'
}

In [ ]:
@lru_cache(maxsize=128)
def names_related(s1: str, s2: str) -> bool:
    buffer = []

    s1 = s1.lower()
    s2 = s2.lower()
    
    if s1 == s2:
        return True
    
    # Edge Case: Bacteria
    # I'm not sure why the kingdom of Escherichia coli
    # is not Bacteria. There is no domain in the data.
    # Anyway, I felt like this would be common enough
    # to warrant a line.
    if (s1 == 'bacteria' and s2 in bacteria) or (s2 == 'bacteria' and s1 in bacteria):
        return True
    
    used_s1 = False
    used_s2 = False
    
    for group in scientific_groups:
        if used_s1 and used_s2:
            break
        
        if not used_s1 and s1 in group['values']:
            buffer.append((s1, group['column']))
            used_s1 = True
            continue

        if not used_s2 and s2 in group['values']:
            buffer.append((s2, group['column']))
            used_s2 = True
            continue
        
    if len(buffer) != 2:
        return False
        
    (t1_val, t1_col), (t2_val, t2_col) = buffer
    
    # Edge Case: Bacteria
    # Well, it's kind of handled.
    if t1_val == 'bacteria' and t2_col in ['col:scientificName', 'col:species']:
        q = f'''
            SELECT LOWER("col:kingdom")
            FROM NameUsage
            WHERE LOWER("{t2_col}") = \'{t2_val}\'
            LIMIT 1
        '''
        output = conn.execute(q).fetchone()
        output = None if not output else output[0].lower()
        if output in {'pseudomonadati', 'bacillati', 'methanobacteriati', 'fusobacteriati', 'thermotogati'}:
            return True
    
    q = f'''
        SELECT 1
        FROM NameUsage
        WHERE LOWER("{t2_col}") = \'{t2_val}\' AND LOWER("{t1_col}") = \'{t1_val}\'
        LIMIT 1
    '''

    output = conn.execute(q)
    return bool(output.fetchone())

In [25]:
def expand_unit(doc: Doc, il_unit: int, ir_unit: int, il_boundary: int, ir_boundary: int, speech: List[str] = [], literals: List[str] = [], include: bool = True, direction: str = 'BOTH', verbose: bool = False):
    UNIT = doc[il_unit:ir_unit+1]
        
    if il_unit > ir_unit:
        print(f"Error: il_unit of {il_unit} greater than ir_unit of {ir_unit}")
        return None
    
    if direction in ['BOTH', 'LEFT'] and il_boundary > il_unit:
        print(f"Error: il_unit of {il_unit} less than il_boundary of {il_boundary}")
        return None
    
    if direction in ['BOTH', 'RIGHT'] and ir_boundary < ir_unit:
        print(f"Error: ir_unit of {ir_unit} greater than ir_boundary of {ir_boundary}")
        return None
    
    # Move Left
    if direction in ['BOTH', 'LEFT']:
        # The indices are inclusive, therefore, when 
        # the condition fails, il_unit will be equal
        # to il_boundary.
        while il_unit > il_boundary:
            # We assume that the current token is allowed,
            # and look to the token to the left.
            l_token = doc[il_unit-1]

            # If the token is invalid, we stop expanding.
            in_set = l_token.pos_ in speech or l_token.lower_ in literals

            # Case 1: include=False, in_set=True
            # If we're not meant to include the defined tokens, and the
            # current token is in that set, we stop expanding.
            # Case 2: include=True, in_set=False
            # If we're meant to include the defined tokens, and the current
            # token is not in that set, we stop expanding.
            # Case 3: include=in_set
            # If we're meant to include the defined tokens, and the current
            # token is in that set, we continue expanding. If we're not meant
            # to include the defined tokens, and the current token is not
            # in that set, we continue expanding.
            if include ^ in_set:
                break
            
            # Else, the left token is valid, and
            # we continue to expand.
            il_unit -= 1

    # Move Right
    if direction in ['BOTH', 'RIGHT']:
        # Likewise, when the condition fails,
        # ir_unit will be equal to the ir_boundary.
        # The ir_boundary is also inclusive.
        while ir_unit < ir_boundary:
            # Assuming that the current token is valid,
            # we look to the right to see if we can
            # expand.
            r_token = doc[ir_unit+1]

            # If the token is invalid, we stop expanding.
            in_set = r_token.pos_ in speech or r_token.lower_ in literals
            if include ^ in_set:
                break

            # Else, the token is valid and
            # we continue.
            ir_unit += 1

    assert il_unit >= il_boundary and ir_unit <= ir_boundary
    expanded_unit = doc[il_unit:ir_unit+1]
        
    return expanded_unit

In [26]:
def contract_unit(doc: Doc, il_unit: int, ir_unit: int, speech: List[str] = [], literals: List[str] = [], include: bool = True, direction: str = 'BOTH', verbose: bool = False):
    UNIT = doc[il_unit:ir_unit+1]
    
    if il_unit > ir_unit:
        print(f"Error: il_unit of {il_unit} greater than ir_unit of {ir_unit}")
        return None
    
    # Move Right
    if direction in ['BOTH', 'LEFT']:
        while il_unit < ir_unit:
            # We must check if the current token is not allowed. If it's
            # not allowed, we contract (remove).
            token = doc[il_unit]

            # include = True means that we want the tokens that match
            # the speech and/or literals in the contracted unit.
            
            # include = False means that we don't want the tokens that
            # match the speech and/or literals in the contracted unit.
            
            # Case 1: include = True, in_set = True
            # We have a token that's meant to be included in the set.
            # However, we're contracting, which means we would end up
            # removing the token if we continue. Therefore, we break.
            
            # Case 2: include = False, in_set = False
            # We have a token that's not in the set which defines the
            # tokens that aren't meant to be included. Therefore, we 
            # have a token that is meant to be included. If we continue,
            # we would end up removing this token. Therefore, we break.
            
            # Default:
            # If we have a token that's in the set (in_set=True) of
            # tokens we're not supposed to include in the contracted 
            # unit (include=False), we need to remove it. Likewise, if
            # we have a token that's not in the set (in_set=False) of
            # tokens to include in the contracted unit (include=True),
            # we need to remove it.
            
            in_set = token.pos_ in speech or token.lower_ in literals
            if include == in_set:
                break

            # The token is valid, thus we continue.
            il_unit += 1

    # Move Left      
    if direction in ['BOTH', 'RIGHT']:
        while ir_unit > il_unit:
            token = doc[ir_unit]

            # The token is invalid and we
            # stop contracting.
            in_set = token.pos_ in speech or token.lower_ in literals
            if include == in_set:
                break

            # The token is valid and we continue.
            ir_unit -= 1

    assert il_unit <= ir_unit
    contracted_unit = doc[il_unit:ir_unit+1]
    
    return contracted_unit

In [27]:
def break_text(text: str, return_type: Literal["Flat", "TextFlat", "Text", "TextAdd"]) -> Union[List[List[Tuple[int, int]]], List[Tuple[int, int]], List[List[str]], List[str]]:
    enclosures = {
        "(":")", 
        "[":"]",
        "{":"}"
    }
    
    # This contains the text that's not inside
    # any enclosure.
    base = []

    # This contains the text that's inside
    # an enclosure.
    groups = []
    
    # This is used for building groups, which can
    # have a nested structure.
    stacks = []
    
    # These are the pairs of characters that
    # define the enclosure (parenthetical).
    openers = list(enclosures.keys())
    closers = list(enclosures.values())
    
    # This contains the opening characters of the groups 
    # that are currently open (e.g. '(', '['). We use it 
    # so that we know whether to open or close a group.
    opened = []
    
    for i, char in enumerate(text):
        # Open Group
        if char in openers:
            stacks.append([])
            opened.append(char)
        # Close Group
        elif opened and char == enclosures.get(opened[-1], ""):
            groups.append(stacks.pop())
            opened.pop()
        # Add to Group
        elif opened:
            stacks[-1].append(i)
        # Add to Base Text
        else:
            base.append(i)
    
    # We close the remaining groups that have not
    # been closed.
    while stacks:
        groups.append(stacks.pop())
        
    # Cluster Groups' Indices
    # A list in the lists of indices (where each list represents a group of text) could have 
    # an interruption (e.g. [0, 1, 2, 10, 15]) because of a parenthetical. So, we cluster the
    # indices in each list to make the output more useful (e.g. [(0, 3), (10, 16)]). As you
    # can see, we've adjusted some indices for ease-of-use.
    lists_of_indices = [*groups, base]        
    lists_of_clustered_indices = []

    for list_of_indices in lists_of_indices:
        if not list_of_indices:
            continue

        # We start off with a single cluster that is made up of the
        # first index. If the next index follows the first index, 
        # we continue the cluster. If it doesn't, we create a new cluster.
        clustered_indices = [[list_of_indices[0], list_of_indices[0] + 1]]
        
        for index in list_of_indices[1:]:
            if clustered_indices[-1][1] == index:
                clustered_indices[-1][1] = index + 1
            else:
                clustered_indices.append([index, index + 1])

        # Add Clustered Indices
        lists_of_clustered_indices.append(clustered_indices)

    if return_type in ["Flat", "TextFlat"]:
        flat_clusters = []
        # We are placing each cluster of indices into one list.
        # This removes the context of the larger parenthetical,
        # but the context may be cumbersome instead of useful.
        for list_of_clustered_indices in lists_of_clustered_indices:
            for clustered_indices in list_of_clustered_indices:
                flat_clusters.append(clustered_indices)
        lists_of_clustered_indices = flat_clusters
    
        if return_type == "TextFlat":
            return [text[cluster[0]:cluster[1]] for cluster in lists_of_clustered_indices]
    
    if return_type in ["Text", "TextAdd"]:
        lists_of_clustered_text = [[text[cluster[0]:cluster[1]] for cluster in clusters] for clusters in lists_of_clustered_indices]
        if return_type == "TextAdd":
            return ["".join(list_of_clustered_text) for list_of_clustered_text in lists_of_clustered_text]
        return lists_of_clustered_text

    return lists_of_clustered_indices

In [28]:
def clean_text(text: str) -> str:
    # 1. Delete Inside and Outside Space
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([?.!,])", r"\1", text)
    text = text.strip()

    # 2. Delete Outside Non-Letters
    while text:
        start_len = len(text)
        # Remove Leading Non-Alphanumeric Character
        if text and not text[0].isalnum():
            text = text[1:]
        # Remove Trailing Non-Alphanumeric Character
        if text and not text[-1].isalnum():
            text = text[:-1]
        # No Changes Made
        if start_len == len(text):
            break
    
    return text

In [29]:
def search_strings_tn(ents: List[Span] | Sequence[Span]) -> Set[str]:
    search_strings = set()
    
    for ent in ents:
        split_text: Any = break_text(ent.text.lower(), return_type="TextAdd")
        split_text = [clean_text(text) for text in split_text]
        search_strings.update(split_text)

        for text in split_text:
            if not text or re.search(r"[^\w\s.-]", text):
                continue
            
            # Add Base Noun
            base = text.split()[-1]
            if not regex.match(r"\w\$.", base):
                search_strings.add(base)

            # Add Singular Version
            s_text = singularize(text)
            if s_text:
                search_strings.add(s_text)

            # Add Plural Version
            p_text = pluralize(text)
            if p_text:
                search_strings.add(p_text)
    
    return search_strings

In [30]:
def find_matches_tn(doc: Doc, search_strings: Set[str]) -> List[Tuple[int, int]]:
    text = doc.text.lower()
    all_matches = []
    for search in search_strings:
        matches = re.finditer(re.escape(search), text, re.IGNORECASE)
        all_matches.extend((match.start(), match.end()) for match in matches)
    return all_matches

In [31]:
def find_matches_db(doc: Doc) -> List[Tuple[int, int]]:
    matches = []
    for r_i, data in all_names.iter(doc.text.lower()):
        key = data['key']
        matches.append((r_i - len(key) + 1, r_i + 1))
    return matches

In [176]:
for group in scientific_groups:
    print(group['label'])

kingdom
phylum
subphylum
class
subclass
order
suborder
superfamily
family
subfamily
genus
subgenus
genericName
specificEpithet
intraspecificEpithet
species
scientificName


In [190]:
import string

def find_ents_from_matches(doc, matches: List[Tuple[int, int]]):
    bad_characters = set([p for p in string.punctuation if p not in ['.', '-']] + [*string.digits])
    
    def is_boundary(char):
        return char.isspace() or char in string.punctuation

    spans = []
    text = doc.text.lower()
    for l_index, r_index in matches:
        # The full word must match, not just a substring inside of it.
        # So, if the species we're looking for is "ant", only "ant"
        # will match -- not "pants" or "antebellum". Therefore, the
        # characters to the left and right of the matched string cannot
        # be letters.
        match = doc.text[l_index:r_index]
        match_lower = text[l_index:r_index]

        l_bound = l_index == 0 or (l_index > 0 and is_boundary(text[l_index-1]))
        r_bound = r_index == len(text) - 1 or (r_index < len(text) and is_boundary(text[r_index]))
        
        if not l_bound or not r_bound or r_index - l_index <= 1:
            continue
        
        # Check 1: No Bad Characters
        match = doc.text[l_index:r_index]
        if not bad_characters.isdisjoint(match):
            continue
        
        # Check 2: Correct Casing, if Scientific
        if data := all_names.get(match_lower, None):
            if data['label'] == 's' and not match[0].isupper():
                continue
        
        span = doc.char_span(l_index, r_index, alignment_mode="expand")
        if not span or not len(span):
            continue
        
        not_sent_start = span.sent.start != span.start

        # Check 3: Correct Casing and Types, Vernacular and Role
        if name_is_vernacular(match_lower) or name_is_role(match_lower):
            if not any(token.pos_ == 'NOUN' for token in span):
                continue

            if not_sent_start and span[0].pos_ == 'NOUN' and span.text[0].isupper():
                continue
        
        # Check 4: Correct Casing, Scientific
        if data := all_names.get(match_lower, None):
            if not_sent_start and name_is_scientific(match_lower) in ['specificEpithet', 'intraspecificEpithet'] and span.text[0].isupper():
                continue
        
        # Expand Species
        # Let's say there's a word like "squirrel". That's a bit ambiguous. 
        # Is it a brown squirrel, a bonobo? If the species is possibly missing
        # information (like an adjective to the left of it), we should expand
        # in order to get a full picture of the species.
        is_short = len(span) == 1 and span[0].pos_ == "NOUN"
        
        # Remove Outer Symbols
        # There are times where a species is identified with a parenthesis
        # nearby. Here, we remove that parenthesis (and any other symbols).
        span = contract_unit(
            doc,
            il_unit=span.start, 
            ir_unit=span.end-1, 
            speech=["ADJ", "PROPN", "NOUN", "X"],
            include=True,
            verbose=False
        )

        if not span or not len(span):
            continue
        
        if is_short:
            span = expand_unit(
                doc,
                il_unit=span.start, 
                ir_unit=span.end-1,
                il_boundary=0,
                ir_boundary=len(doc),
                speech=["ADJ", "PROPN", "NOUN"],
                literals=["-"],
                include=True,
                direction="LEFT",
                verbose=False
            )

            if not span:
                continue
            
            # Remove Outer Symbols (Again)
            # The previous expansion contains a literal.
            # There might not be a need for that literal.
            # To handle that case, we contract.
            span = contract_unit(
                doc,
                il_unit=span.start,
                ir_unit=span.end-1,
                speech=["ADJ", "PROPN", "NOUN", "X"],
                include=True,
                verbose=False
            )
            
            if not span or not len(span):
                continue
            
        # A species must have a noun or a
        # proper noun. This may help discard
        # bad results.
        noun_found = False
        for token in span:
            if token.pos_ in ["NOUN", "PROPN", "X"]:
                noun_found = True
                break
        
        if not noun_found:
            continue

        # The names I'd like to identify should
        # not have any numbers or odd characters.
        if re.search(r"[^\w\s.-]", span.text):
            continue
        
        # Added
        spans.append(span)

    spans = filter_spans(spans)
    return spans

In [33]:
def find_ents_tn(doc: Doc) -> List[Span]:
    searches = search_strings_tn(doc.ents)
    matches = find_matches_tn(doc, searches)
    return find_ents_from_matches(doc, matches)

In [ ]:
def find_ents_db(doc: Doc) -> List[Span]:
    matches = find_matches_db(doc)
    return find_ents_from_matches(doc, matches)

In [224]:
def find_ents(doc: Doc) -> List[Span]:
    ents_tn = find_ents_tn(doc)
    ents_db = find_ents_db(doc)
    ents = filter_spans([*ents_tn, *ents_db])

    if not ents:
        return ents
    
    # Merge Consecutive Spans (Intervals)
    ents.sort(key=lambda ent: ent.start)
    
    merged = ents and [ents[0]]
    for current in ents[1:]:
        previous = merged[-1]
        if previous.end >= current.start:
            end = max(previous.end, current.end)
            merged[-1] = doc[previous.start:end]
        else:
            merged.append(current)
    
    return merged

In [103]:
def find_substitutions(spans: List[Span], doc: Doc) -> Dict[str, List[str]]:
    spans = [*spans]
    spans.sort(key=lambda span: span.start)
    
    token_to_span = {}
    for span in spans:
        for token in span:
            token_to_span[token] = span
    
    # The text provides information about 
    # names that will be used interchangeably,
    # like "predatory crab (Carcinus maenas)".
    substitutions: Dict[str, List[str]] = {}

    def add_substitutions(a, b):
        notation_a = name_is_taxonomic_notation(a)
        notation_b = name_is_taxonomic_notation(b)

        a = a if notation_a else a.lower()
        b = b if notation_b else b.lower()
                
        if a not in substitutions:
            substitutions[a] = []
        substitutions[a].append(b)

        return

    tracking = None
    for token in doc:
        # Neighboring Spans
        # We're only using the last token of the span.
        if token in token_to_span and token_to_span[token][-1] == token:
            token_span = token_to_span[token]
            token_n1 = token.i + 1 <= len(doc) - 1 and token.nbor(1)
            token_n2 = token.i + 2 <= len(doc) - 1 and token.nbor(2)
            
            same_sent_n2 = token and token_n2 and token.sent == token_n2.sent
            
            # Use Case: "Diptera: Tephritidae"
            if token_n1 and token_n2 and same_sent_n2 and token_n1.lower_ in ['.', ':'] and token_n2 in token_to_span:
                add_substitutions(token_span.text, token_to_span[token_n2].text)
        
        # Dealing with Parentheses
        # 1. Opening Parentheses
        if token.text == "(":
            tracking = token

        # 2. Closing Parentheses
        if tracking and token.text == ")":
            # Spans in Parentheses
            tracked_spans = set()
            for i in range(tracking.i + 1, token.i):
                token = doc[i]
                if token in token_to_span:
                    tracked_spans.add(token_to_span[token])

            # Span to Left of Parentheses
            tracking_token = tracking.i != 0 and tracking.nbor(-1)
            if tracking_token and tracking_token in token_to_span:
                tracking_span = token_to_span[tracking_token]
                tracking_span_text = tracking_span.text

                for span in tracked_spans:
                    span_text = span.text
                    add_substitutions(tracking_span_text, span_text)
            
            tracking = None
    
    return substitutions

In [ ]:
class BaseEntityGroup(TypedDict):
    labels: Set[str]
    names: Set[str]
    lowers: Set[str]
    subs: Set[str]
    subs_mapped: Dict[str, Set[str]]
    forms: Set[str]
    taxa: Set[str]
    scientific: Set[str]
    vernacular: Set[str]

EntityGroup = BaseEntityGroup | Dict[str, Set[str] | Dict[str, Set[str]]]

In [83]:
def create_entity_group(name: str) -> EntityGroup:
    name_lower = name.lower()
    
    # Taxonomic Name
    name_taxonomic = name if is_taxonomic_name(name) else is_taxonomic_notation(name)
    name_is_notation = name and name != name_taxonomic
    name_taxonomic_lower = None if not name_taxonomic else name_taxonomic.lower()

    # Get Labels
    label = None
    if name_is_vernacular(name_lower):
        label = 'v'
    elif name_is_role(name_lower):
        label = 'r'
    elif name_is_scientific(name_taxonomic_lower or name_lower):
        label = 's'
    else:
        label = 'v'
    
    # Get Substitutions
    subs = mapped_names.get(name_taxonomic_lower or name_lower, set())
    
    # Get Different Forms of Name
    def clean(s: str) -> str:
        s = re.sub(rf'[{string.punctuation}]', ' ', s)
        s = re.sub(r'\s+', ' ', s)
        return s
    
    # Get Forms
    forms = set()
    
    # Casing matters if the name is that
    # of taxonomic notation. For example,
    # Salmo trutta l. != Salmo trutta L.
    if name_is_notation:
        forms.add(name)
        forms.add(clean(name))
    else:
        forms.add(name_lower)
        forms.add(clean(name_lower))
    
    if name_taxonomic_lower:
        forms.add(name_taxonomic_lower)
    
    if label == 's' and name_taxonomic and (inflections := abbreviate(name_taxonomic)):
        for inflection in inflections:
            inflection_lower = inflection.lower()
            forms.add(inflection_lower)
            forms.add(clean(inflection_lower))
    else:
        inflections = get_inflections(name)
        for inflection in inflections:
            inflection = inflection.lower()
            forms.add(inflection)
            forms.add(clean(inflection))
    
    return {
        f'{label}': {name},
        'labels': {label},
        'names': {name},
        'forms': forms,
        'lowers': {name_lower},
        'subs': subs,
        'subs_mapped': {form: subs for form in forms},
        'taxa': forms if label == 's' and name_is_taxa(name) else set(),
        'scientific': forms if label == 'sf' else set(),
        'vernacular': forms if label == 'v' else set(),
    }

In [39]:
def merge_entity_group(a: EntityGroup, b: EntityGroup) -> EntityGroup:
    merged = {}
    
    keys = set([*a.keys(), *b.keys()])
    for key in keys:
        a_val = a.get(key, set())
        b_val = b.get(key, set())
        merged[key] = a_val | b_val # type: ignore
    
    return merged

In [111]:
def merge_entity_groups(merge: Dict[int, Set[int]], groups: List[EntityGroup]) -> List[EntityGroup]:
    merge_items = list(merge.items())
    merge_items.sort(key=lambda item: len(item[1]), reverse=True)

    for item in merge_items:
        print(item)
    
    for source_i, targets_i in merge_items:
        if not groups[source_i]:
            continue
        
        for target_i in targets_i:
            if source_i == target_i:
                continue
            
            if not groups[target_i]:
                continue
            
            groups[target_i] = merge_entity_group(
                groups[target_i], 
                groups[source_i]
            )
        
        if [i for i in targets_i if groups[i]]:
            groups[source_i] = typing.cast(EntityGroup, None)

    return [group for group in groups if group]

In [41]:
def group_by_contains(groups: List[EntityGroup]) -> List[EntityGroup]:
    merge = {i: set() for i in range(len(groups))}
    
    i = 0
    while i < len(groups):
        j = i + 1
        while j < len(groups):
            forms_i: Any = groups[i]['forms']
            forms_j: Any = groups[j]['forms']

            if not forms_i.isdisjoint(forms_j):
                merge[i].add(j)
            else:
                for form_i in forms_i:
                    for form_j in forms_j:
                        if re.search(rf"\b{form_i}\b", form_j):
                            merge[i].add(j)
                        if re.search(rf"\b{form_j}\b", form_i):
                            merge[j].add(i)
            
            j += 1
        i += 1

    return merge_entity_groups(merge, groups)

In [90]:
def group_by_internal_subs(groups: List[EntityGroup], subs: Dict[str, List[str]]) -> List[EntityGroup]:
    merge = {i: set() for i in range(len(groups))}
    
    for k, v in subs.items():
        sources = [i for i, group in enumerate(groups) if group['forms'] & {k}] # type: ignore
        targets = [i for i, group in enumerate(groups) if i not in sources and group['forms'] & set(v)] # type: ignore

        print(k, v, sources, targets)

        # Swap
        # Either 'sources' or 'targets' will have only 1
        # element.
        if len(sources) > len(targets):
            sources, targets = targets, sources

        if not sources:
            continue
        
        merge[sources[0]].update(targets)
    
    return merge_entity_groups(merge, groups)

In [232]:
def group_by_external_subs(groups: List[EntityGroup]) -> List[EntityGroup]:
    merge = {i: set() for i in range(len(groups))}
    
    i = 0
    while i < len(groups):
        j = i + 1
        while j < len(groups):
            if i in merge.get(j, []):
                j += 1
                continue

            found = False

            pairs = [
                ['taxa', 'vernacular'],
                ['vernacular', 'taxa'],
                ['scientific', 'vernacular'],
                ['vernacular', 'scientific']
            ]
            
            for l1, l2 in pairs:
                A: Set[str] = groups[i][l1] - groups[j][l1] # type: ignore
                B: Set[str] = groups[j][l2] - groups[i][l2] # type: ignore

                # We do not want to merge two different species because of a
                # similar ancestor. Alas, the 'taxa' group cannot have any species.
                if l1 == 'taxa' and any(is_taxonomic_name(n) or is_taxonomic_notation(n) for n in groups[i]['names']):
                    continue

                if l2 == 'taxa' and any(is_taxonomic_name(n) or is_taxonomic_notation(n) for n in groups[j]['names']):
                    continue

                subs_mapped_a: Dict[str, Set[str]] = groups[i]['subs_mapped'] # type: ignore
                subs_mapped_b: Dict[str, Set[str]] = groups[j]['subs_mapped'] # type: ignore

                A_subs = set().union(*[subs_mapped_a[a] for a in A if subs_mapped_a[a]]) or A
                B_subs = set().union(*[subs_mapped_b[b] for b in B if subs_mapped_b[b]]) or B
  
                if not A_subs.isdisjoint(B_subs):
                    print(l1, l2, i, j, A_subs & B_subs)
                    merge[i].add(j)
                    merge[j].add(i)
                    found = True
                    break
            
            if not found:
                A: Set[str] = groups[i]['forms'] - groups[j]['forms'] # type: ignore
                B: Set[str] = groups[j]['forms'] - groups[i]['forms'] # type: ignore

                print('Scientific v Vernacular')
                for a in A:
                    for b in B:
                        print(a, b)
                        if names_related(a, b):
                            print(f'{a} and {b} are Related')
                            merge[i].add(j)
                            merge[j].add(i)
                            found = True
                            break
                    
                    if found:
                        break
            
            j += 1
        i += 1

    return merge_entity_groups(merge, groups)

In [84]:
def group_ents(ents: List[Span], doc: Doc, verbose=False) -> List[EntityGroup]:
    mapped_ents = {}
    for ent in ents:
        mapped_ents[ent.text.lower()] = ent
    
    distinct_ents = [*mapped_ents.values()]
    distinct_ents.sort(key=lambda ent: ent.text.lower())
    groups: Any = [[ent] for ent in distinct_ents]

    t0 = time.time()
    
    i = 0
    while i < len(groups):
        groups[i] = create_entity_group(groups[i][0].text)
        i += 1
    type(typing.cast(List[EntityGroup], groups))

    if verbose:
        print(f'Create Groups: {time.time() - t0}s')
        for g_i, group in enumerate(groups):
            print(g_i, group['forms'], group['labels'])
        print()
        print()
    
    
    t0 = time.time()
    groups = group_by_contains(groups)    
    if verbose:
        print(f'Group by Contains: {time.time() - t0}s')
        for g_i, group in enumerate(groups):
            print(g_i, group['forms'])
        print()
        print()
    

    t0 = time.time()
    subs = find_substitutions(ents, doc)
    print(f'Subs: {subs}')
    
    groups = group_by_internal_subs(groups, subs)
    if verbose:
        print(f'Group by Internal Subs: {time.time() - t0}s')
        for g_i, group in enumerate(groups):
            print(g_i, group['forms'])
        print()
        print()

    
    t0 = time.time()
    groups = group_by_external_subs(groups)
    if verbose:
        print(f'Group by External Subs: {time.time() - t0}s')
        for g_i, group in enumerate(groups):
            print(g_i, group['forms'])
        print()
        print()

    if verbose:
        print()
        for g_i, group in enumerate(groups):
            print(g_i, group['forms'])
        print()
        print()

    return groups

In [45]:
spacy.require_gpu()  # type: ignore
nlp = TaxoNERD(prefer_gpu=True).load(
    model="en_ner_eco_biobert", 
    exclude=["tok2vec", "parser", "lemmatizer"]
)

In [46]:
df = pd.read_excel("./benchmarks/Benchmark-03-09.xlsx")
texts = df.Abstract.tolist()
number_texts = len(texts)

In [47]:
papers_3 = df[df['Score'] == 3]
papers_0 = df[(df['Score'] < 3) & (df['Score'] >= 0)]

In [219]:
# type: ignore
import unicodedata
from tqdm.auto import tqdm

bad = []
times = []

for i, row in tqdm(papers_3.iterrows()):
    txt = unicodedata.normalize("NFKC", row.Abstract)
    txt = regex.sub(r'(\p{Ll})(\p{Lu})', r'\1 \2', txt)
    txt = regex.sub(r'([\.,:;\)\]\}])([\p{Lu}\p{Ll}])', r'\1 \2', txt)
    txt = regex.sub(r'([\p{Lu}\p{Ll}])([\(\[\{]])', r'\1 \2', txt)
    
    doc = nlp(txt)

    t0 = time.time()
    ents = find_ents(doc)
    
    # Store Counts
    counts = {}
    for ent in ents:
        counts[ent.text.lower()] = counts.get(ent.text.lower(), 0) + 1
    
    # Calculate Counts of Group
    groups: List[Any] = group_ents(ents, doc, verbose=False)
    for group in groups:
        count = sum([counts[lower] for lower in group['lowers']])
        group['count'] = count

    cond_1 = len(groups) >= 3
    cond_2 = len([group for group in groups if group['count'] >= 2]) >= 3
    flag = cond_1 and cond_2
    t1 = time.time()
    
    times.append((i, t1-t0))
    if flag != True:
        bad.append(i)

0it [00:00, ?it/s]

(1, {6, 14})
(0, {8})
(2, set())
(3, set())
(4, set())
(5, set())
(6, set())
(7, set())
(8, set())
(9, set())
(10, set())
(11, set())
(12, set())
(13, set())
(14, set())
Subs: {'small bluegills': ['lepomis macrochirus'], 'largemouth bass': ['micropterus salmoides'], 'cladocerans': ['diaphanosoma', 'daphnia', 'ceriodaphnia']}
small bluegills ['lepomis macrochirus'] [12] [7]
largemouth bass ['micropterus salmoides'] [6] [8]
cladocerans ['diaphanosoma', 'daphnia', 'ceriodaphnia'] [1] [0, 2, 3]
(1, {0, 2, 3})
(6, {8})
(12, {7})
(0, set())
(2, set())
(3, set())
(4, set())
(5, set())
(7, set())
(8, set())
(9, set())
(10, set())
(11, set())
taxa vernacular 0 3 {'animalia'}
taxa vernacular 0 5 {'animalia'}
taxa vernacular 0 6 {'animalia'}
taxa vernacular 1 3 {'animalia'}
taxa vernacular 1 5 {'animalia'}
taxa vernacular 1 6 {'animalia'}
(0, {3, 5, 6})
(1, {3, 5, 6})
(3, {0, 1})
(5, {0, 1})
(6, {0, 1})
(2, set())
(4, set())
(7, set())
(8, set())
(9, set())


2it [00:01,  1.17it/s]

(6, {8, 7})
(0, {1})
(14, {11})
(1, set())
(2, set())
(3, set())
(4, set())
(5, set())
(7, set())
(8, set())
(9, set())
(10, set())
(11, set())
(12, set())
(13, set())
Subs: {'sculpins': ['cottus bairdi'], 'perlid stoneflies': ['agnetina capitata'], 'ephemeroptera prey': ['ephemerella subvaria', 'baetis tricaudatus']}
sculpins ['cottus bairdi'] [11] [4]
perlid stoneflies ['agnetina capitata'] [9] [0]
ephemeroptera prey ['ephemerella subvaria', 'baetis tricaudatus'] [7] [2, 6]
(7, {2, 6})
(9, {0})
(11, {4})
(0, set())
(1, set())
(2, set())
(3, set())
(4, set())
(5, set())
(6, set())
(8, set())
(10, set())
(0, set())
(1, set())
(2, set())
(3, set())
(4, set())
(5, set())
(6, set())
(7, set())
(8, set())


3it [00:02,  1.36it/s]

...

(0, set())
(1, set())
(2, set())
(3, set())
(4, set())
Subs: {'green frog': ['rana clamitans']}
green frog ['rana clamitans'] [0] [3]
(0, {3})
(1, set())
(2, set())
(3, set())
(4, set())
(0, set())
(1, set())
(2, set())
(3, set())


82it [01:03,  3.96it/s]

(6, {0, 9, 4, 7})
(7, {0, 9, 4})
(8, {1, 5})
(10, {1})
(0, set())
(1, set())
(2, set())
(3, set())
(4, set())
(5, set())
(9, set())
(11, set())
Subs: {}
(0, set())
(1, set())
(2, set())
(3, set())
(4, set())
(5, set())
(6, set())
(7, set())
(0, set())
(1, set())
(2, set())
(3, set())
(4, set())
(5, set())
(6, set())
(7, set())


83it [01:03,  1.31it/s]

(4, {2, 5, 6})
(6, {2, 5})
(10, {8, 5})
(1, {3})
(11, {12})
(0, set())
(2, set())
(3, set())
(5, set())
(7, set())
(8, set())
(9, set())
(12, set())
Subs: {'gastrointestinal helminths': ['graphidium strigosum', 'trichostrongylus retortaeformis'], 'european rabbits': ['oryctolagus cuniculus']}
gastrointestinal helminths ['graphidium strigosum', 'trichostrongylus retortaeformis'] [1] [2, 7]
european rabbits ['oryctolagus cuniculus'] [0] [4]
(1, {2, 7})
(0, {4})
(2, set())
(3, set())
(4, set())
(5, set())
(6, set())
(7, set())
(0, set())
(1, set())
(2, set())
(3, set())
(4, set())
(5, set())


In [220]:
times.sort(key=lambda t: t[1], reverse=True)
print(f'Bad: {bad}')
print(f'Times')
for t in times:
    print(f'{t}')

Bad: [16, 34, 85, 94, 111, 126]
Times
(30, 24.14476180076599)
(19, 4.255370140075684)
(111, 2.9332568645477295)
(103, 2.5370383262634277)
(5, 1.5087640285491943)
(3, 1.4792530536651611)
(0, 0.49927735328674316)
(22, 0.4822700023651123)
(14, 0.4539322853088379)
(10, 0.4060037136077881)
(2, 0.40276050567626953)
(26, 0.37670230865478516)
(12, 0.37096381187438965)
(7, 0.36829638481140137)
(82, 0.34441709518432617)
(28, 0.30659008026123047)
(252, 0.2987360954284668)
(35, 0.282397985458374)
(27, 0.27503490447998047)
(89, 0.27295351028442383)
(42, 0.26720261573791504)
(9, 0.26630520820617676)
(257, 0.2653203010559082)
(31, 0.26375365257263184)
(1, 0.26155734062194824)
(18, 0.24262666702270508)
(21, 0.23841452598571777)
(277, 0.23552632331848145)
(4, 0.2320859432220459)
(88, 0.23203206062316895)
(83, 0.23163175582885742)
(13, 0.22758007049560547)
(110, 0.22722697257995605)
(43, 0.22504305839538574)
(38, 0.21753597259521484)
(95, 0.21269583702087402)
(269, 0.21239304542541504)
(32, 0.2087891101

In [300]:
# type: ignore
import unicodedata

i = 94

txt = unicodedata.normalize("NFKC", df.loc[i, 'Abstract'])
txt = regex.sub(r'(\p{Ll})(\p{Lu})', r'\1 \2', txt)
txt = regex.sub(r'([\.,:;\)\]\}])([\p{Lu}\p{Ll}])', r'\1 \2', txt)
txt = regex.sub(r'([\p{Lu}\p{Ll}])([\(\[\{]])', r'\1 \2', txt)

# byThe -> by The
# L.l -> L. l
# :Words -> : Words
# 
print('Unprocessed')
print(df.loc[i, 'Abstract'])
print()
print('Processed')
print(txt)
print()
print()

doc = nlp(txt)
print('doc.ents', doc.ents)
print()
print()

ents_tn = find_ents_tn(doc)
print(ents_tn)
print(f'TN Entities: {set({ent.text.lower() for ent in ents_tn})}')
print()
print()

ents_db = find_ents_db(doc)
print(ents_db)
print(f'DB Entities: {set({ent.text.lower() for ent in ents_db})}')
print()
print()

ents = filter_spans([*ents_tn, *ents_db])

# Merge Consecutive Spans (Intervals)
ents.sort(key=lambda ent: ent.start)
print(list((ent.start, ent.end) for ent in ents))

merged = ents and [ents[0]]
for current in ents[1:]:
    previous = merged[-1]
    if previous.end >= current.start:
        end = max(previous.end, current.end)
        merged[-1] = doc[previous.start:end]
    else:
        merged.append(current)

ents = merged
print(list((ent.start, ent.end) for ent in ents))
print()
print()

print('Entities')
counts = {}
for ent in ents:
    print(ent)
    counts[ent.text.lower()] = counts.get(ent.text.lower(), 0) + 1
print()
print()

groups = group_ents(ents, doc, verbose=True)
for group in groups:
    count = sum([counts[lower] for lower in group['lowers']])
    group['count'] = count
    print(group['lowers'], group['count'])

cond_1 = len(groups) >= 3
cond_2 = len([group for group in groups if group['count'] >= 2]) >= 3

print(cond_1)
print(cond_2)

Unprocessed
Gut environments harbour dense microbial ecosystems in which plasmids are widely distributed. Plasmids facilitate the exchange of genetic material among microorganisms while enabling the transfer of a diverse array of accessory functions. However, their precise impact on microbial community composition and function remains largely unexplored. Here we identify a prevalent bacterial toxin and a plasmid-encoded resistance mechanism that mediates the interaction between Lactobacilli and Enterococci. This plasmid is widespread across ecosystems, including the rumen and human gut microbiota. Biochemical characterization of the plasmid revealed a defence mechanism against reuterin, a toxin produced by various gut microbes, such as Limosilactobacillus reuteri. Using a targeted metabolomic approach, we find reuterin to be prevalent across rumen ecosystems with impacts on microbial community structure. Enterococcus strains carrying the protective plasmid were isolated and their inter

In [204]:
for token in doc:
    if 'consumer' in token.text.lower():
        print(1)

1


In [231]:
# name_is_vernacular('bacteria')
mapped_names['bacteria']

KeyError: 'bacteria'

In [234]:
name_is_scientific('escherichia coli')

'species'

In [236]:
names_related('bacteria', 'escherichia coli')

False

In [ ]:
def get_flag(doc: Doc):
    ents = find_ents(doc)
    
    # Store Counts
    counts = {}
    for ent in ents:
        counts[ent.text.lower()] = counts.get(ent.text.lower(), 0) + 1
    
    # Calculate Counts of Group
    groups: List[Any] = group_ents(ents, doc)
    for group in groups:
        count = sum([counts[lower] for lower in group['lowers']])
        group['count'] = count

    cond_1 = len(groups) >= 3
    cond_2 = len([group for group in groups if group['count'] >= 3]) >= 2
    
    return (bool(cond_1 and cond_2), ents, groups)

In [ ]:
def next_suffix(regex, files):
    max_suffix = -1
    for file in files:
        match = re.match(regex, file)
        if match:
            suffix = int(match.group(1))
            max_suffix = max(max_suffix, suffix)
    return max_suffix

In [ ]:
def save(*, mask, counts, errors, suffix=None):
    outputs = {
        "counts": counts,
        "mask": mask,
        "errors": errors
    }
    
    if not suffix:
        files = [f for f in listdir('./') if isfile(join('./', f))]
        suffix = next_suffix(r"ScreenByEntitiesOutput-(\d+).pickle", files)
    
    with open(f'ScreenByEntitiesOutput-{suffix or 0}.pickle', 'wb') as file:
        pickle.dump(outputs, file)

In [ ]:
def load(suffix=None):
    outputs = {
        "counts": {None: 0, True: 0, False: 0},
        "mask": [],
        "errors": {}
    }

    if not suffix:
        files = [f for f in listdir('./') if isfile(join('./', f))]
        suffix = next_suffix(r"ScreenByEntitiesOutput-(\d+).pickle", files)
    
    with open(f'ScreenByEntitiesOutput-{suffix or 0}.pickle', 'rb') as file:
        outputs = pickle.load(file)
    
    return outputs

In [ ]:
# type: ignore
outputs = load()
mask = outputs[mask]
counts = outputs[counts]
errors = outputs[errors]

i = len(mask)
for doc in tqdm(nlp.pipe(texts, batch_size=16), total=number_texts):
    flag = None

    # Auto-Save
    if (i + 1) % 20:
        save(mask=mask, counts=counts, errors=errors)
            
    try:
        flag = get_flag(doc)[0]
    except Exception as e:
        errors[i] = e

    counts[flag] += 1
    mask.append(flag)

    clear_output(wait=True)
    print(f"{i+1}/{number_texts})")
    print(f"Number Included: {counts[True]}")
    print(f"Number Excluded: {counts[False]}")
    print(f"Number Errors: {counts[None]}")

    i += 1